# Notebook 3: RoBERTa Fine-Tuning, Ablation & Evaluation
### Product Review Rating Prediction — Transformer Model

**Architecture**: `roberta-base` → Mean Pooling (last hidden layer, masked) → Dropout (p=0.1) → Linear (5 classes)

> **Prerequisites**: Run `notebook1_preprocessing.ipynb` first.

**Ablation Study**: Mean Pooling vs. First-Token (`<s>`) Pooling, and Max Length 128 vs. 256

In [ ]:
!pip install -q transformers[torch] datasets accelerate scikit-learn matplotlib seaborn

In [ ]:
import os, json, random, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset, load_from_disk
from transformers import (
    RobertaModel,
    RobertaTokenizerFast,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

SEEDS       = [42, 123]
NUM_CLASSES = 5
MODEL_NAME  = 'roberta-base'
USE_AMP     = torch.cuda.is_available()
print('Mixed precision (AMP):', USE_AMP)

## 1. Load Tokenized Datasets

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)
collator  = DataCollatorWithPadding(tokenizer, return_tensors='pt')

train_ds     = load_from_disk('processed/roberta_train')
val_ds       = load_from_disk('processed/roberta_val')
test_ds      = load_from_disk('processed/roberta_test')
train_ds_128 = load_from_disk('processed/roberta_train_128')
val_ds_128   = load_from_disk('processed/roberta_val_128')
test_ds_128  = load_from_disk('processed/roberta_test_128')

print('Train:', len(train_ds), '| Val:', len(val_ds), '| Test:', len(test_ds))
print('Columns:', train_ds.column_names)

In [ ]:
def make_roberta_loaders(train_data, val_data, test_data, batch_size=32, num_workers=2):
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                              collate_fn=collator, num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_data,   batch_size=batch_size, shuffle=False,
                              collate_fn=collator, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=False,
                              collate_fn=collator, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

## 2. Model Architecture

In [ ]:
class RoBERTaClassifier(nn.Module):
    """
    RoBERTa fine-tuned for 5-class rating prediction.
    pooling: 'mean' (masked mean over last hidden states) or 'cls' (first-token).
    """

    def __init__(self, model_name='roberta-base', num_classes=5, dropout=0.1, pooling='mean'):
        super().__init__()
        assert pooling in ('mean', 'cls')
        self.roberta    = RobertaModel.from_pretrained(model_name)
        self.pooling    = pooling
        hidden_size     = self.roberta.config.hidden_size  # 768
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def mean_pool(self, last_hidden, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).float()
        summed        = (last_hidden * mask_expanded).sum(1)
        counts        = mask_expanded.sum(1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids, attention_mask):
        outputs     = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        pooled      = self.mean_pool(last_hidden, attention_mask) if self.pooling == 'mean' \
                      else last_hidden[:, 0, :]
        return self.classifier(self.dropout(pooled))


_m   = RoBERTaClassifier().to(DEVICE)
_ids = torch.zeros(2, 16, dtype=torch.long).to(DEVICE)
_msk = torch.ones(2, 16, dtype=torch.long).to(DEVICE)
print('Output shape:', _m(_ids, _msk).shape)
del _m, _ids, _msk

## 3. Training Loop

In [ ]:
def get_label_key(batch):
    return 'labels' if 'labels' in batch else 'label'


def evaluate_roberta(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    criterion  = nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in loader:
            lk        = get_label_key(batch)
            labels    = batch.pop(lk).to(device)
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch['attention_mask'].to(device)
            logits    = model(input_ids, attn_mask)
            loss      = criterion(logits, labels)
            total_loss += loss.item() * len(labels)
            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    n   = len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    mf1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    mae = float(np.mean(np.abs(np.array(all_preds) - np.array(all_labels))))
    return total_loss / n, acc, mf1, mae, all_preds, all_labels


def train_roberta(
    seed, pooling='mean', max_len=256, lr=2e-5, weight_decay=0.01,
    batch_size=16, grad_accum=2, max_epochs=5, patience=2,
    warmup_ratio=0.06, dropout=0.1, num_workers=2, label_str='',
):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    tr, va, te = (train_ds_128, val_ds_128, test_ds_128) if max_len == 128 \
                 else (train_ds, val_ds, test_ds)
    train_loader, val_loader, test_loader = make_roberta_loaders(
        tr, va, te, batch_size=batch_size, num_workers=num_workers
    )

    model = RoBERTaClassifier(
        model_name=MODEL_NAME, num_classes=NUM_CLASSES, dropout=dropout, pooling=pooling
    ).to(DEVICE)

    no_decay      = ['bias', 'LayerNorm.weight']
    optim_groups  = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optimizer     = torch.optim.AdamW(optim_groups, lr=lr)
    total_steps   = (len(train_loader) // grad_accum) * max_epochs
    warmup_steps  = int(total_steps * warmup_ratio)
    scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    criterion     = nn.CrossEntropyLoss()
    scaler        = torch.cuda.amp.GradScaler() if USE_AMP else None

    best_mf1 = -1.0; best_state = None; no_improve = 0
    history  = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_mf1': []}
    tag      = label_str or f'{pooling}_len{max_len}_seed{seed}'

    for epoch in range(1, max_epochs + 1):
        model.train(); epoch_loss = 0.0; t0 = time.time(); optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            lk        = get_label_key(batch)
            labels    = batch.pop(lk).to(DEVICE)
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)

            if USE_AMP:
                with torch.cuda.amp.autocast():
                    logits = model(input_ids, attn_mask)
                    loss   = criterion(logits, labels) / grad_accum
                scaler.scale(loss).backward()
            else:
                loss = criterion(model(input_ids, attn_mask), labels) / grad_accum
                loss.backward()

            epoch_loss += loss.item() * grad_accum * len(labels)

            if (step + 1) % grad_accum == 0:
                if USE_AMP:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer); scaler.update()
                else:
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step(); optimizer.zero_grad()

        train_loss = epoch_loss / len(train_loader.dataset)
        val_loss, val_acc, val_mf1, val_mae, _, _ = evaluate_roberta(model, val_loader, DEVICE)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_mf1'].append(val_mf1)

        print(f'[{tag}] Ep {epoch:02d} [{time.time()-t0:.0f}s] | '
              f'TrLoss={train_loss:.4f} ValLoss={val_loss:.4f} '
              f'ValAcc={val_acc:.4f} ValMF1={val_mf1:.4f} ValMAE={val_mae:.4f}')

        if val_mf1 > best_mf1:
            best_mf1 = val_mf1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
            torch.save(best_state, f'roberta_best_{tag}.pt')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping at epoch {epoch}.')
                break

    model.load_state_dict(best_state)
    test_loss, test_acc, test_mf1, test_mae, test_preds, test_true = \
        evaluate_roberta(model, test_loader, DEVICE)
    print(f'[{tag}] TEST  Acc={test_acc:.4f} MF1={test_mf1:.4f} MAE={test_mae:.4f}')
    return history, test_acc, test_mf1, test_mae, test_preds, test_true, model

## 4. Main Training (Mean Pooling, 256 tokens, 2 Seeds)

In [ ]:
roberta_results = {}

for seed in SEEDS:
    print(f'\n{"="*70}\nRoBERTa (mean, 256) — SEED {seed}\n{"="*70}')
    history, acc, mf1, mae, preds, trues, model = train_roberta(
        seed=seed, pooling='mean', max_len=256, label_str=f'mean256_seed{seed}'
    )
    roberta_results[seed] = dict(history=history, acc=acc, mf1=mf1, mae=mae, preds=preds, trues=trues)

r_accs = [roberta_results[s]['acc'] for s in SEEDS]
r_mf1s = [roberta_results[s]['mf1'] for s in SEEDS]
r_maes = [roberta_results[s]['mae'] for s in SEEDS]
print(f'\n=== RoBERTa Main Results (mean pooling) ===')
print(f'Accuracy: {np.mean(r_accs):.4f} ± {np.std(r_accs):.4f}')
print(f'Macro F1: {np.mean(r_mf1s):.4f} ± {np.std(r_mf1s):.4f}')
print(f'MAE:      {np.mean(r_maes):.4f} ± {np.std(r_maes):.4f}')

## 5. Ablation A: Mean vs. CLS Pooling

In [ ]:
print('\n=== Ablation A: CLS Pooling, 256 tokens, seed=42 ===')
_, cls_acc, cls_mf1, cls_mae, cls_preds, cls_trues, _ = train_roberta(
    seed=42, pooling='cls', max_len=256, label_str='cls256_seed42'
)
mean_acc42 = roberta_results[42]['acc']
mean_mf142 = roberta_results[42]['mf1']
mean_mae42 = roberta_results[42]['mae']

print(f'\nPooling       | Acc    | MF1    | MAE')
print(f'Mean (256)    | {mean_acc42:.4f} | {mean_mf142:.4f} | {mean_mae42:.4f}')
print(f'CLS  (256)    | {cls_acc:.4f} | {cls_mf1:.4f} | {cls_mae:.4f}')

## 6. Ablation B: Max Length 128 vs. 256

In [ ]:
print('\n=== Ablation B: Max Length 128, mean pooling, seed=42 ===')
_, len128_acc, len128_mf1, len128_mae, len128_preds, len128_trues, _ = train_roberta(
    seed=42, pooling='mean', max_len=128, label_str='mean128_seed42'
)
print(f'\nMax Len | Acc    | MF1    | MAE')
print(f'256     | {mean_acc42:.4f} | {mean_mf142:.4f} | {mean_mae42:.4f}')
print(f'128     | {len128_acc:.4f} | {len128_mf1:.4f} | {len128_mae:.4f}')

## 7. Training Curves

In [ ]:
best_seed_r = max(SEEDS, key=lambda s: roberta_results[s]['mf1'])
history_r   = roberta_results[best_seed_r]['history']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history_r['train_loss']) + 1)

axes[0].plot(epochs, history_r['train_loss'], 'o-', label='Train Loss')
axes[0].plot(epochs, history_r['val_loss'],   's-', label='Val Loss')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='RoBERTa — Loss Curves')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, history_r['val_acc'], 'o-', label='Val Accuracy')
axes[1].plot(epochs, history_r['val_mf1'], 's-', label='Val Macro F1')
axes[1].set(xlabel='Epoch', ylabel='Score', title='RoBERTa — Validation Metrics')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('roberta_training_curves.png', dpi=150)
plt.show()

## 8. Detailed Evaluation

In [ ]:
best_preds_r = roberta_results[best_seed_r]['preds']
best_trues_r = roberta_results[best_seed_r]['trues']

print('Classification Report (RoBERTa):')
print(classification_report(best_trues_r, best_preds_r,
                             target_names=[f'{i+1}-star' for i in range(5)]))

cm_r      = confusion_matrix(best_trues_r, best_preds_r)
cm_r_norm = cm_r.astype(float) / cm_r.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_r_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=[f'{i+1}★' for i in range(5)],
            yticklabels=[f'{i+1}★' for i in range(5)])
ax.set(xlabel='Predicted Rating', ylabel='True Rating', title='RoBERTa — Normalized Confusion Matrix')
plt.tight_layout()
plt.savefig('roberta_confusion_matrix.png', dpi=150)
plt.show()

preds_arr = np.array(best_preds_r); trues_arr = np.array(best_trues_r)
diffs     = np.abs(preds_arr - trues_arr)
print('\nOrdinal Error Analysis (RoBERTa):')
for label, cond in [('Correct (0)', diffs==0), ('Adjacent (1)', diffs==1),
                     ('Near-miss (2)', diffs==2), ('Distant (>=3)', diffs>=3)]:
    print(f'  {label}: {cond.mean():.3f}')

## 9. Comparative Analysis: BiGRU vs. RoBERTa

In [ ]:
bigru_preds   = np.load('bigru_test_preds.npy')
bigru_trues   = np.load('bigru_test_trues.npy')
with open('bigru_results.json') as f:
    bigru_summary = json.load(f)

print(f'{"Model":<28} {"Accuracy":>14} {"Macro F1":>14} {"MAE":>10}')
print('-'*70)
print(f'{"BiGRU (Baseline)":<28} '
      f'{bigru_summary["acc_mean"]:.4f}±{bigru_summary["acc_std"]:.4f}  '
      f'{bigru_summary["mf1_mean"]:.4f}±{bigru_summary["mf1_std"]:.4f}  '
      f'{bigru_summary["mae_mean"]:.4f}±{bigru_summary["mae_std"]:.4f}')
print(f'{"RoBERTa (Mean Pool, 256)":<28} '
      f'{np.mean(r_accs):.4f}±{np.std(r_accs):.4f}  '
      f'{np.mean(r_mf1s):.4f}±{np.std(r_mf1s):.4f}  '
      f'{np.mean(r_maes):.4f}±{np.std(r_maes):.4f}')

In [ ]:
bigru_cm   = confusion_matrix(bigru_trues, bigru_preds)
bigru_cm_n = bigru_cm.astype(float) / bigru_cm.sum(axis=1, keepdims=True)
labels     = [f'{i+1}★' for i in range(5)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(bigru_cm_n,  annot=True, fmt='.2f', cmap='Oranges', ax=axes[0],
            xticklabels=labels, yticklabels=labels)
axes[0].set(xlabel='Predicted', ylabel='True', title='BiGRU — Confusion Matrix')

sns.heatmap(cm_r_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
            xticklabels=labels, yticklabels=labels)
axes[1].set(xlabel='Predicted', ylabel='True', title='RoBERTa — Confusion Matrix')

plt.tight_layout()
plt.savefig('comparison_confusion_matrices.png', dpi=150)
plt.show()

## 10. Qualitative Analysis

In [ ]:
raw_test = load_dataset('SetFit/amazon_reviews_multi_en', split='test')

def get_text(sample):
    return (sample.get('text') or ((sample.get('review_title') or '') + ' ' + (sample.get('review_body') or ''))).strip()

n         = min(len(bigru_preds), len(best_preds_r), len(raw_test))
bigru_p   = np.array(bigru_preds[:n])
robt_p    = np.array(best_preds_r[:n])
true_l    = np.array(bigru_trues[:n])

bf_rw = np.where((bigru_p != true_l) & (robt_p == true_l))[0]
rf_bw = np.where((robt_p  != true_l) & (bigru_p == true_l))[0]
print(f'BiGRU fails, RoBERTa wins: {len(bf_rw)}')
print(f'RoBERTa fails, BiGRU wins: {len(rf_bw)}')

In [ ]:
def show_examples(indices, title, max_n=5):
    rng     = np.random.default_rng(99)
    chosen  = rng.choice(indices, size=min(max_n, len(indices)), replace=False)
    print(f'\n{"="*70}\n{title}\n{"="*70}')
    for i, idx in enumerate(chosen):
        sample = raw_test[int(idx)]
        print(f'\n[{i+1}] idx={idx} | True={true_l[idx]+1}★ | BiGRU={bigru_p[idx]+1}★ | RoBERTa={robt_p[idx]+1}★')
        print(f'Text: {get_text(sample)[:350]}')

show_examples(bf_rw, 'BiGRU FAILS — RoBERTa SUCCEEDS (min 5)', max_n=5)
show_examples(rf_bw, 'RoBERTa FAILS — BiGRU SUCCEEDS (min 2)', max_n=2)

## 11. Token Importance Visualization

In [ ]:
model.eval()

def visualize_token_importance(model, tokenizer, text, true_label, device=DEVICE):
    """Gradient-norm saliency over RoBERTa input embeddings."""
    inputs = tokenizer(text, return_tensors='pt', max_length=256, truncation=True).to(device)

    embeddings = model.roberta.embeddings(inputs['input_ids'])
    embeddings.retain_grad()

    outputs = model.roberta(inputs_embeds=embeddings, attention_mask=inputs['attention_mask'])
    lh      = outputs.last_hidden_state
    mask    = inputs['attention_mask'].unsqueeze(-1).float()
    pooled  = (lh * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    logits  = model.classifier(model.dropout(pooled))

    logits[0, true_label].backward()

    grad       = embeddings.grad[0].detach().cpu().norm(dim=-1).numpy()
    tokens     = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].cpu().tolist())
    n          = len(tokens)
    importance = (grad[:n] - grad[:n].min()) / (np.ptp(grad[:n]) + 1e-9)

    fig, ax = plt.subplots(figsize=(max(10, n // 3), 3))
    ax.bar(range(n), importance, color='steelblue')
    ax.set_xticks(range(n))
    ax.set_xticklabels(tokens, rotation=90, fontsize=8)
    pred_class = logits.argmax().item() + 1
    ax.set(ylabel='Importance', title=f'Token Importance | True: {true_label+1}★ | Pred: {pred_class}★')
    plt.tight_layout()
    plt.savefig('roberta_token_importance.png', dpi=150)
    plt.show()


if len(bf_rw) > 0:
    vis_idx    = int(bf_rw[0])
    vis_text   = get_text(raw_test[vis_idx])
    vis_label  = int(true_l[vis_idx])
    print(f'Example idx={vis_idx} | True={vis_label+1}★')
    print(f'Text: {vis_text[:200]}')
    visualize_token_importance(model, tokenizer, vis_text, vis_label)

## 12. Save All Results

In [ ]:
roberta_summary = {
    'model': 'RoBERTa-base (mean pooling, 256 tokens)',
    'seeds': SEEDS,
    'acc_mean': float(np.mean(r_accs)), 'acc_std': float(np.std(r_accs)),
    'mf1_mean': float(np.mean(r_mf1s)), 'mf1_std': float(np.std(r_mf1s)),
    'mae_mean': float(np.mean(r_maes)), 'mae_std': float(np.std(r_maes)),
    'ablation': {
        'mean_256_seed42': {'acc': float(mean_acc42), 'mf1': float(mean_mf142), 'mae': float(mean_mae42)},
        'cls_256_seed42':  {'acc': float(cls_acc),    'mf1': float(cls_mf1),    'mae': float(cls_mae)},
        'mean_128_seed42': {'acc': float(len128_acc), 'mf1': float(len128_mf1), 'mae': float(len128_mae)},
    }
}

with open('roberta_results.json', 'w') as f:
    json.dump(roberta_summary, f, indent=2)

np.save('roberta_test_preds.npy', np.array(best_preds_r))
np.save('roberta_test_trues.npy', np.array(best_trues_r))

print('All results saved.')
print(json.dumps(roberta_summary, indent=2))